In [ ]:
%pip install -q transformers torch gradio plotly pandas

In [ ]:
from transformers import AutoTokenizer,AutoModel
import torch

Model_name="distilbert-base-uncased"

tokenizer=AutoTokenizer.from_pretrained(Model_name)

model=AutoModel.from_pretrained(
    Model_name,
    output_attentions=True
)

model.eval()
print("Model loaded successfully")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully


In [ ]:
model.config  #model loaded successfully

DistilBertConfig {
  "activation": "gelu",
  "architectures": [
    "DistilBertForMaskedLM"
  ],
  "attention_dropout": 0.1,
  "bos_token_id": null,
  "dim": 768,
  "dropout": 0.1,
  "dtype": "float32",
  "eos_token_id": null,
  "hidden_dim": 3072,
  "initializer_range": 0.02,
  "max_position_embeddings": 512,
  "model_type": "distilbert",
  "n_heads": 12,
  "n_layers": 6,
  "output_attentions": true,
  "pad_token_id": 0,
  "qa_dropout": 0.1,
  "seq_classif_dropout": 0.2,
  "sinusoidal_pos_embds": false,
  "tie_weights_": true,
  "tie_word_embeddings": true,
  "transformers_version": "5.10.2",
  "vocab_size": 30522
}

### Understanding the Attention Tensor

In [8]:
text="When it rains look for Rainbow, when it's dark look for stars "

inputs=tokenizer(
    text,
    return_tensors="pt"
)

with torch.no_grad():
    outputs=outputs=model(**inputs)

attentions=outputs.attentions


print(len(attentions))
print(attentions[0].shape)

6
torch.Size([1, 12, 17, 17])


In [ ]:
##building an helper function

import numpy as np

def extract_attention(text,layer_idx,head_idx):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=64
    )
    with torch.no_grad():
        outputs = model(**inputs)

    attention = outputs.attentions[layer_idx]
    attention = attention[0, head_idx]
    tokens = tokenizer.convert_ids_to_tokens(
        inputs["input_ids"][0]
    )

    return tokens, attention.cpu().numpy()

In [11]:
tokens, attn = extract_attention(
    "When it rains look for Rainbow, when it's dark look for stars ",
    0,
    0
)

print(tokens)
print(attn.shape)

['[CLS]', 'when', 'it', 'rains', 'look', 'for', 'rainbow', ',', 'when', 'it', "'", 's', 'dark', 'look', 'for', 'stars', '[SEP]']
(17, 17)


In [ ]:
import pandas as pd
import plotly.express as px

def create_heatmap(
    text,
    layer_idx,
    head_idx
):

    tokens, attn = extract_attention(
        text,
        layer_idx,
        head_idx
    )

    df = pd.DataFrame(
        attn,
        index=tokens,
        columns=tokens
    )

    fig = px.imshow(
        df,
        text_auto=".2f",
        aspect="auto",
        title=f"Layer {layer_idx} | Head {head_idx}"
    )

    return fig

: 

In [18]:
import gradio as gr

MAX_LAYER = 5
MAX_HEAD = 11

In [19]:
def visualize(
    text,
    layer,
    head
):

    return create_heatmap(
        text,
        layer,
        head
    )

In [20]:
demo = gr.Interface(
    fn=visualize,

    inputs=[
        gr.Textbox(
            value="When it rains look for Rainbow, when it's dark look for stars ",
            label="Input Text"
        ),

        gr.Slider(
            0,
            MAX_LAYER,
            value=0,
            step=1,
            label="Layer"
        ),

        gr.Slider(
            0,
            MAX_HEAD,
            value=0,
            step=1,
            label="Head"
        )
    ],

    outputs=gr.Plot(),

    title="Transformer Attention Explorer",

    description="""
    Visualize attention weights from DistilBERT.
    Explore different layers and attention heads.
    """
)

In [21]:
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5f71d4e77f875efdd9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Created dataset file at: .gradio/flagged/dataset1.csv
